In [ ]:
import time
from bs4 import BeautifulSoup
import requests
import pandas as pd
import numpy as np

# Function to extract Product Title
def get_title(soup):
    try:
        title = soup.find("span", attrs={"id": 'productTitle'}).text.strip()
    except AttributeError:
        title = ""
    return title

# Function to extract Product Price
def get_price(soup):
    try:
        price = soup.find("span", attrs={'id': 'priceblock_ourprice'}).text.strip()
    except AttributeError:
        try:
            price = soup.find("span", attrs={'id': 'priceblock_dealprice'}).text.strip()
        except:
            price = ""
    return price

# Function to extract Product Rating
def get_rating(soup):
    try:
        rating = soup.find("span", attrs={'class': 'a-icon-alt'}).text.strip()
    except AttributeError:
        rating = ""
    return rating

# Function to extract Number of User Reviews
def get_review_count(soup):
    try:
        review_count = soup.find("span", attrs={'id': 'acrCustomerReviewText'}).text.strip()
    except AttributeError:
        review_count = ""
    return review_count

# Function to extract Availability Status
def get_availability(soup):
    try:
        available = soup.find("div", attrs={'id': 'availability'}).find("span").text.strip()
    except AttributeError:
        available = "Not Available"
    return available

# Main Function
if __name__ == '__main__':
    HEADERS = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36',
        'Accept-Language': 'en-US, en;q=0.5'
    }

    # Base URL for electronics category
    base_url = "https://www.amazon.com/s?k=Electronics&crid=2R2YUBOBYJLRX&sprefix=electronics%2Caps%2C101&ref=nb_sb_noss_1"

    # Dictionary to store product details
    products = {"title": [], "price": [], "rating": [], "reviews": [], "availability": []}

    # Number of pages to scrape (adjust to cover more products)
    total_pages = 20 # Adjust this value based on requirements and testing

    # Loop through pages
    for page in range(1, total_pages + 1):
        print(f"Scraping page {page}...")

        # Request the page
        url = base_url.format(page=page)
        webpage = requests.get(url, headers=HEADERS)
        soup = BeautifulSoup(webpage.content, "html.parser")

        # Fetch links to individual product pages
        links = soup.find_all("a", attrs={'class': 'a-link-normal s-no-outline'})
        links_list = [link.get('href') for link in links]

        for link in links_list:
            # Check if link already starts with "https://www.amazon.com"
            if link.startswith("https://www.amazon.com"):
                product_url = link  # Use the link directly
            else:
                product_url = "https://www.amazon.com" + link  # Concatenate if needed
            product_page = requests.get(product_url, headers=HEADERS)
            product_soup = BeautifulSoup(product_page.content, "html.parser")
            # Extract product details
            products['title'].append(get_title(product_soup))
            products['price'].append(get_price(product_soup))
            products['rating'].append(get_rating(product_soup))
            products['reviews'].append(get_review_count(product_soup))
            products['availability'].append(get_availability(product_soup))

            # Throttle requests to prevent blocking
            time.sleep(1)

        # Save intermediate results periodically
        if page % 10 == 0:  # Save after every 10 pages
            temp_df = pd.DataFrame.from_dict(products)
            temp_df.to_csv(f"amazon_electronics_page_{page}.csv", index=False, header=True)
            print(f"Saved results up to page {page}.")

        # Add delay between pages
        time.sleep(5)

    # Create final DataFrame and save to CSV
    final_df = pd.DataFrame.from_dict(products)
    final_df.to_csv("amazon_electronics_full.csv", index=False, header=True)
    print("Scraping complete. Data saved to 'amazon_electronics_full.csv'.")


Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...
Scraping page 6...
Scraping page 7...
Scraping page 8...
Scraping page 9...
Scraping page 10...
Saved results up to page 10.
Scraping page 11...
Scraping page 12...
Scraping page 13...
Scraping page 14...
Scraping page 15...
Scraping page 16...
Scraping page 17...
Scraping page 18...
Scraping page 19...


ConnectionError: HTTPSConnectionPool(host='www.amazon.comhttps', port=443): Max retries exceeded with url: /aax-us-iad.amazon.com/x/c/JO3iqoQ2hbZ2cz3IoJ6r6ZQAAAGUdWtFRAEAAAH2AQBvbm9fdHhuX2JpZDMgICBvbm9fdHhuX2ltcDIgICCBiitf/clv1_CEuOPUxokZA0iHrVBeoO2iL8SV5nVIgnjJwG-jqT6ftx4X8a9TKbUvw18nHE8xvWjq0yzpNb3XTPsunPNWcAWEVALiG9H-TvFXMfXkpwRF24Ap54lEfB8IkGXHyXURUFlnfjQIl6FCHP2G5XBb0A4Y64GwKhsQDHc5EJRNpsUhiYi_2DhRIgVEG2MJusK7AjRioRNNeDh8xCBywyfYFaa6JQzNWz1tQoGAWUZugh0QYNsYMulremNDcyil-xxVwBHWfqObnPtRUr30EeO509Jz3lOVCbozbBIv-27fDFCrQSEkm_1uS-9AqaewyGIraJHZ1mNo8OP7XzmLbJIdCZBpt91u20MFNSxsrmYj5qhFSB3SYQd0Mxbbk/https://www.amazon.com/Projector-Video-Projector-Multimedia-Compatible-Smartphone/dp/B07MTCMHZX/ref=sxbs_sbv_search_btf?content-id=amzn1.sym.2f0a8989-0b67-47e7-b61e-9e3ef9908602%3Aamzn1.sym.2f0a8989-0b67-47e7-b61e-9e3ef9908602&crid=2R2YUBOBYJLRX&cv_ct_cx=Electronics&keywords=Electronics&pd_rd_i=B07MTCMHZX&pd_rd_r=6f53d75f-c909-4747-a61b-d9246c98c1e6&pd_rd_w=lt1M7&pd_rd_wg=lJxWL&pf_rd_p=2f0a898

Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...
Scraping page 6...
Scraping page 7...
Scraping page 8...
Scraping page 9...
Scraping page 10...
Saved results up to page 10.
Scraping page 11...
Scraping page 12...
Scraping page 13...
Scraping page 14...
Scraping page 15...
Scraping page 16...
Scraping page 17...
Scraping page 18...
Scraping page 19...
Scraping page 20...
Saved results up to page 20.
Scraping page 21...
Scraping page 22...
Scraping page 23...
Scraping page 24...
Scraping page 25...
Scraping page 26...
Scraping page 27...
Scraping page 28...
Scraping page 29...
Scraping page 30...
Saved results up to page 30.
Scraping page 31...
Scraping page 32...
Scraping page 33...
Scraping page 34...
Scraping page 35...
Scraping page 36...
Scraping page 37...
Scraping page 38...
Scraping page 39...
Scraping page 40...
Saved results up to page 40.
Scraping page 41...
Scraping page 42...
Scraping page 43...
Scraping page 44...
Scraping page